In [ ]:
import os
print(os.getcwd())

In [ ]:
print(os.listdir())

In [ ]:
import joblib
import database as db
import preprocessing
import xgboost as xgb
import pandas as pd
import numpy as np

In [ ]:
engine = db.get_engine()
db.test_connection(engine)

raw_df = db.fetch_recently_updated_customers(engine)

if raw_df.empty:
    print("No rows updated in the last 5 minutes.")
    exit()

print("Rows sent to model:", len(raw_df))
raw_df.head()

In [ ]:
package = joblib.load("preprocessing_package.pkl")
features_df = preprocessing.transform(raw_df, package)

In [ ]:
print(features_df.shape)
print(features_df.columns.tolist())
print(features_df.isnull().sum().sum())  

In [ ]:
import xgboost as xgb

MODEL_PATH = "xgboost_model.json"

model = xgb.XGBClassifier()

model.load_model(MODEL_PATH)

print("XGBoost model loaded successfully.")

In [ ]:
expected_features = len(
    package["feature_order"]
)

actual_features = features_df.shape[1]

print(
    f"Expected features: {expected_features}"
)

print(
    f"Actual features  : {actual_features}"
)

if actual_features != expected_features:
    raise ValueError(
        "Feature count mismatch between "
        "preprocessing package and XGBoost model."
    )

In [ ]:
churn_probability = model.predict_proba(
    features_df
)[:, 1]

In [ ]:

threshold = 0.50

churn_prediction = (
    churn_probability >= threshold
).astype(int)

In [ ]:
import pandas as pd
import numpy as np

print(pd.__name__)
print(np.__name__)

In [ ]:
# ============================================================
# CREATE RESULT DATAFRAME
# ============================================================

result = pd.DataFrame(index=raw_df.index)

# Find customer ID case-insensitively
customer_id_col = next(
    (
        col
        for col in raw_df.columns
        if col.strip().lower() == "customerid"
    ),
    None
)

if customer_id_col is None:
    raise ValueError(
        "customerID column not found in PostgreSQL data."
    )

result["customerID"] = raw_df[
    customer_id_col
].values

# Add prediction probability
result["Churn_Probability"] = churn_probability

# Add 0/1 prediction
result["Churn_Prediction"] = churn_prediction

# Add readable label
result["Churn_Label"] = result[
    "Churn_Prediction"
].map({
    0: "No Churn",
    1: "Churn"
})

print("\n======================================")
print("CHURN PREDICTION RESULTS")
print("======================================")

print(
    result.head(20).to_string(
        index=False
    )
)

In [ ]:
print("\n======================================")
print("SUMMARY")
print("======================================")

print(
    "Total customers:",
    len(result)
)

print(
    "Predicted Churn:",
    (result["Churn_Prediction"] == 1).sum()
)

print(
    "Predicted No Churn:",
    (result["Churn_Prediction"] == 0).sum()
)

In [ ]:
# ============================================================
# SAVE CHURN PREDICTIONS BACK TO POSTGRESQL
# ============================================================

OUTPUT_TABLE = "churn_predictions"

result.to_sql(
    OUTPUT_TABLE,
    engine,
    schema="public",
    if_exists="replace",
    index=False
)

print(
    f"\nPrediction results written to PostgreSQL table: {OUTPUT_TABLE}"
)

In [ ]:
check = pd.read_sql(
    "SELECT * FROM public.churn_predictions LIMIT 10",
    engine
)

print(check)

In [ ]:
import xgboost as xgb

LTV_MODEL_PATH = "ltv_model.json"

ltv_model = xgb.XGBRegressor()

ltv_model.load_model(LTV_MODEL_PATH)

print("LTV model loaded successfully.")


In [ ]:
predicted_ltv = ltv_model.predict(
    features_df
)

print("LTV prediction completed.")


In [ ]:
result["Predicted_LTV"] = predicted_ltv


In [ ]:
print("\n======================================")
print("CHURN + LTV RESULTS")
print("======================================")

print(
    result.head(20).to_string(
        index=False
    )
)

In [ ]:
result["Predicted_LTV"] = predicted_ltv